In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd
import scanpy as sc

In [2]:
import sys
sys.path.append('../../FUNCTIONS/')
from methods import NMF_to_module, module_to_json

In [3]:
!pwd
%cd Cell/
!pwd

/hdd/edward/the-project/parameter_tuning/NMF
/hdd/edward/the-project/parameter_tuning/NMF/Cell
/hdd/edward/the-project/parameter_tuning/NMF/Cell


In [4]:
# Load your data matrix (samples x genes)
# Ensure it's a NumPy array or pandas DataFrame
adata = sc.read_h5ad("../../../results/Cell/discrete_cancer_stages.h5ad")
adata

AnnData object with n_obs × n_vars = 115774 × 12939
    obs: 'sample_ID', 'Gender', 'INSS_stage', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'cell_type', '_scvi_batch', '_scvi_labels', 'leiden_scVI', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std'
    uns: 'INSS_stage_colors', '_scvi_manager_uuid', '_scvi_uuid', 'discrete_cancer_stage_colors', 'hvg', 'leiden_scVI', 'leiden_scVI_colors', 'leiden_scVI_sizes', 'log1p', 'neighbors', 'paga', 'sample_ID_colors', 'scrublet', 'umap'
    obsm: 'X_scVI', 'X_umap', '_scvi_extra_continuous_covs'
    layers: 'scvi_log', 'scvi_normalized', 'scvi_scale'
    obsp: 'connectivities', 'distances'

In [5]:
adata_4S = adata[adata.obs.discrete_cancer_stage == "4S"]
adata_I = adata[adata.obs.discrete_cancer_stage == "I"]
adata_III = adata[adata.obs.discrete_cancer_stage == "III"]
adata_IV = adata[adata.obs.discrete_cancer_stage == "IV"]
adata_4S

View of AnnData object with n_obs × n_vars = 9111 × 12939
    obs: 'sample_ID', 'Gender', 'INSS_stage', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'cell_type', '_scvi_batch', '_scvi_labels', 'leiden_scVI', 'discrete_cancer_stage'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std'
    uns: 'INSS_stage_colors', '_scvi_manager_uuid', '_scvi_uuid', 'discrete_cancer_stage_colors', 'hvg', 'leiden_scVI', 'leiden_scVI_colors', 'leiden_scVI_sizes', 'log1p', 'neighbors', 'paga', 'sample_ID_colors', 'scrublet', 'umap'
    obsm: 'X_scVI', 'X_umap', '_scvi_extra_continuous_covs'
    layers: 'scvi_log', 'scvi_normalized', 'scvi_scale'
    obsp: 'connectivities', 'distances'

4S

In [6]:
data_4S = pd.DataFrame(adata_4S.layers["scvi_log"], index=adata_4S.obs_names, columns=adata_4S.var_names)
data_4S

Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,MATR3,RGS5
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AACACGTAGCGCTCCA.1,0.117779,0.207004,0.062645,0.027983,0.007688,0.084976,0.038014,0.031309,0.021099,0.048666,...,0.022521,0.010148,0.005971,0.003456,0.010245,0.185664,0.082429,0.008154,0.009808,1.787761
T44_AACCATGGTCGATTGT.1,0.114513,0.212297,0.056350,0.024811,0.008003,0.067652,0.037309,0.030162,0.016578,0.041567,...,0.017777,0.009541,0.005648,0.004055,0.010374,0.168756,0.074406,0.007086,0.009222,1.822530
T44_AACCGCGCAGTCGTGC.1,0.143959,0.202033,0.055459,0.022813,0.010484,0.108309,0.030245,0.017870,0.041686,0.069236,...,0.028582,0.011240,0.008323,0.005477,0.010065,0.272724,0.085935,0.009180,0.010645,2.080249
T44_AACTCAGGTGCGGTAA.1,0.167583,0.185374,0.072528,0.021188,0.013228,0.095900,0.045566,0.024819,0.040607,0.084826,...,0.029234,0.015266,0.008186,0.005998,0.012753,0.283838,0.077284,0.010193,0.013616,1.891863
T44_AACTGGTGTGTTGGGA.1,0.111846,0.151710,0.062685,0.019705,0.010370,0.113933,0.040740,0.017119,0.046687,0.087225,...,0.023780,0.013733,0.007734,0.004598,0.012713,0.266793,0.075005,0.010068,0.012423,2.211435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTTATGCTCTGCTGTC.1,0.147554,0.441396,0.068053,0.088651,0.027734,0.131112,0.042450,0.034982,0.013190,0.038375,...,0.029598,0.008959,0.011536,0.013897,0.012570,0.145869,0.111658,0.010242,0.019190,1.612637
T40_TTTCCTCAGATGCCAG.1,0.128449,0.472242,0.068920,0.034879,0.037634,0.176414,0.062225,0.039750,0.024819,0.055453,...,0.031699,0.009913,0.014264,0.018611,0.014537,0.178586,0.116881,0.013406,0.022552,2.184139
T40_TTTGCGCAGCTACCGC.1,0.115588,0.573728,0.080294,0.053624,0.038048,0.153981,0.056442,0.053334,0.016050,0.058655,...,0.032469,0.010005,0.013072,0.020760,0.015320,0.158535,0.136642,0.014412,0.021113,2.184520


In [7]:
num_of_modules = [20,40,60,80,100,120,170,220,270]

for i in num_of_modules:
    # Initialize the NMF model
    model = NMF(n_components=i, init='nndsvd', random_state=123456, max_iter=1000)

    # Perform NMF
    W = model.fit_transform(data_4S)  # W: Samples × Components
    H = model.components_         # H: Components × Genes

    # # Save results
    # np.savetxt("W_matrix.csv", W, delimiter=",")
    # np.savetxt("H_matrix.csv", H, delimiter=",")
    module_4S = NMF_to_module(stage='4S', modules=H, genes=data_4S.columns, min_module_size=4)
    module_to_json(module_dict=module_4S, file_path=f"module_4S_{i}.json")

/hdd/edward/.conda/envs/NB/lib/python3.8/site-packages/sklearn/decomposition/_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


20
Number of modules: 20


/hdd/edward/.conda/envs/NB/lib/python3.8/site-packages/sklearn/decomposition/_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


38
Number of modules: 36


/hdd/edward/.conda/envs/NB/lib/python3.8/site-packages/sklearn/decomposition/_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


...

/hdd/edward/.conda/envs/NB/lib/python3.8/site-packages/sklearn/decomposition/_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


45
Number of modules: 35


/hdd/edward/.conda/envs/NB/lib/python3.8/site-packages/sklearn/decomposition/_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


47
Number of modules: 28
36
Number of modules: 21


/hdd/edward/.conda/envs/NB/lib/python3.8/site-packages/sklearn/decomposition/_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


I

In [17]:
data_I = pd.DataFrame(adata_I.layers["scvi_log"], index=adata_I.obs_names, columns=adata_I.var_names)
data_I

Symbol,TSPAN6,DPM1,SCYL3,C1orf112,FGR,FUCA2,GCLC,NFYA,STPG1,NIPAL3,...,SEPSECS-AS1,LINC01144,ADORA3,BISPR,PRNCR1,LINC01481,CYB561D2,CCDC7,MATR3,RGS5
Cell_barcode,,,,,,,,,,,,,,,,,,,,,
T44_AAACCTGAGAGGTTGC.1,0.128050,0.140652,0.080902,0.025638,0.013584,0.100803,0.046459,0.022789,0.041882,0.120709,...,0.033836,0.019171,0.007148,0.005269,0.016824,0.298898,0.072412,0.013041,0.013972,2.007362
T44_AAACCTGAGAGTGAGA.1,0.101546,0.109528,0.062917,0.017231,0.010815,0.092403,0.044756,0.015744,0.041342,0.098031,...,0.020953,0.014917,0.006820,0.005170,0.014650,0.255010,0.061181,0.010310,0.014844,2.146356
T44_AAACCTGAGGCATGTG.1,0.125425,0.146424,0.070615,0.021486,0.012645,0.104837,0.035065,0.016561,0.038813,0.112497,...,0.036598,0.017381,0.008282,0.005117,0.015163,0.308354,0.080154,0.013429,0.013483,2.101646
T44_AAACCTGAGTGTACCT.1,0.143190,0.121734,0.056516,0.014646,0.012271,0.093070,0.037508,0.013392,0.041921,0.070634,...,0.016618,0.014891,0.013192,0.006903,0.011128,0.245671,0.083397,0.009572,0.014492,2.059853
T44_AAACCTGCAATGCCAT.1,0.106379,0.144439,0.064377,0.017019,0.009616,0.094652,0.053437,0.021363,0.034417,0.092099,...,0.022067,0.012800,0.006841,0.005627,0.011460,0.256871,0.089309,0.010954,0.011589,2.191989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T40_TTGGCAAGTCAGATAA.1,0.123533,0.460738,0.069624,0.150854,0.028424,0.203173,0.033974,0.037468,0.019574,0.042135,...,0.040880,0.009897,0.013986,0.011908,0.014184,0.150704,0.104540,0.013908,0.017696,1.590467
T40_TTGTAGGCAGCGAACA.1,0.112014,0.579877,0.061707,0.079144,0.036903,0.125004,0.046963,0.050878,0.014679,0.044751,...,0.021767,0.007375,0.013644,0.015390,0.020946,0.215494,0.070603,0.011617,0.021829,1.799805
T40_TTTGCGCGTCAGAAGC.1,0.077789,0.495869,0.047323,0.106668,0.042440,0.162525,0.045299,0.056460,0.022435,0.035724,...,0.037173,0.010589,0.016257,0.021022,0.016292,0.114849,0.103121,0.010218,0.019198,2.187807


In [ ]:
for i in num_of_modules:
    # Initialize the NMF model
    model = NMF(n_components=i, init='nndsvd', random_state=123456, max_iter=1000)

    # Perform NMF
    W = model.fit_transform(data_I)  # W: Samples × Components
    H = model.components_         # H: Components × Genes

    # # Save results
    # np.savetxt("W_matrix.csv", W, delimiter=",")
    # np.savetxt("H_matrix.csv", H, delimiter=",")
    module_I = NMF_to_module(stage='I', modules=H, genes=data_I.columns, min_module_size=4)
    module_to_json(module_dict=module_I, file_path=f"module_I_{i}.json")

III

In [ ]:
data_III = pd.DataFrame(adata_III.layers["scvi_log"], index=adata_III.obs_names, columns=adata_III.var_names)
data_III

In [ ]:
for i in num_of_modules:
    # Initialize the NMF model
    model = NMF(n_components=i, init='nndsvd', random_state=123456, max_iter=1000)

    # Perform NMF
    W = model.fit_transform(data_III)  # W: Samples × Components
    H = model.components_         # H: Components × Genes

    # # Save results
    # np.savetxt("W_matrix.csv", W, delimiter=",")
    # np.savetxt("H_matrix.csv", H, delimiter=",")
    module_III = NMF_to_module(stage='III', modules=H, genes=data_III.columns, min_module_size=4)
    module_to_json(module_dict=module_III, file_path=f"module_III_{i}.json")

IV

In [ ]:
data_IV = pd.DataFrame(adata_IV.layers["scvi_log"], index=adata_IV.obs_names, columns=adata_IV.var_names)
data_IV

In [ ]:
for i in num_of_modules:
    # Initialize the NMF model
    model = NMF(n_components=i, init='nndsvd', random_state=123456, max_iter=1000)

    # Perform NMF
    W = model.fit_transform(data_IV)  # W: Samples × Components
    H = model.components_         # H: Components × Genes

    # # Save results
    # np.savetxt("W_matrix.csv", W, delimiter=",")
    # np.savetxt("H_matrix.csv", H, delimiter=",")
    module_IV = NMF_to_module(stage='IV', modules=H, genes=data_IV.columns, min_module_size=4)
    module_to_json(module_dict=module_IV, file_path=f"module_IV_{i}.json")